In [1]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))


In [2]:
refine_plus_export_pool = Path("refine_plus_export_pool")
combined_db_export_pool = Path("combined_db_export_pool")
export_folder: Path = combined_db_export_pool / "pass_00"

In [3]:
import os
from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from boulder_statistics.refinement_plus.refinement_chunking import ChunkingTools

lfs_to_join: list[LazyFrame] = [
    pl.scan_parquet(refine_plus_export_pool / table_name)
    for table_name in os.listdir(refine_plus_export_pool)
] + [
    dp.combined_atlas,
    dp.combined_mask.unique(
    subset=["i", "j", "face"], keep="first" # Sometimes duplicates can occur but are rare
    ).rename({
        "lod_level": "detection_lod_level",
        "lod_code": "detection_lod_code",
        "row_id" : "boulder_id"
    })
]

for lf in lfs_to_join:
    print(lf.collect_schema().names())

c:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\.venv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


['i', 'j', 'face', 'area']
['i', 'j', 'face', 'TIR detailed_survey TIR detailed_survey tri_num', 'TIR detailed_survey band depth 350', 'TIR detailed_survey band depth 440', 'TIR detailed_survey slope 1000', 'TIR detailed_survey ratio 1000', 'TIR detailed_survey sigma band depth 350', 'TIR detailed_survey sigma band depth 440', 'TIR detailed_survey sigma slope 1000', 'TIR detailed_survey sigma ratio 1000', 'TIR detailed_survey tri_num']
['i', 'j', 'face', 'TIR recona TIR recona tri_num', 'TIR recona band depth 350', 'TIR recona band depth 440', 'TIR recona slope 1000', 'TIR recona ratio 1000', 'TIR recona sigma band depth 350', 'TIR recona sigma band depth 440', 'TIR recona sigma slope 1000', 'TIR recona sigma ratio 1000', 'TIR recona tri_num']
['i', 'j', 'face', 'TIR reconb TIR reconb tri_num', 'TIR reconb band depth 350', 'TIR reconb band depth 440', 'TIR reconb slope 1000', 'TIR reconb ratio 1000', 'TIR reconb sigma band depth 350', 'TIR reconb sigma band depth 440', 'TIR reconb sigm

In [4]:
ChunkingTools.join_in_chunks(
    export_folder = export_folder,
    lfs_to_join = lfs_to_join,
    chunks = QCubeChunk.generate(depth=4),
    n_jobs = 8,
)

Joining:   0%|          | 0/1536 [00:00<?, ?it/s]

DuplicateError: column with name 'band depth_right' already exists

You may want to try:
- renaming the column prior to joining
- using the `suffix` parameter to specify a suffix different to the default one ('_right')

In [ ]:
pl.scan_parquet(export_folder).filter(pl.col("i") == 1).collect()

i,j,face,area,detailed_survey band depth 350,detailed_survey band depth 440,detailed_survey slope 1000,detailed_survey ratio 1000,detailed_survey sigma band depth 350,detailed_survey sigma band depth 440,detailed_survey sigma slope 1000,detailed_survey sigma ratio 1000,detailed_survey tri_num,recona band depth 350,recona band depth 440,recona slope 1000,recona ratio 1000,recona sigma band depth 350,recona sigma band depth 440,recona sigma slope 1000,recona sigma ratio 1000,recona tri_num,reconb band depth 350,reconb band depth 440,reconb slope 1000,reconb ratio 1000,reconb sigma band depth 350,reconb sigma band depth 440,reconb sigma slope 1000,reconb sigma ratio 1000,reconb tri_num,detailed_survey band depth,detailed_survey reflectance,detailed_survey slope1 poly,detailed_survey slope2 poly,detailed_survey sigma band depth,detailed_survey sigma reflectance,detailed_survey sigma slope1 poly,detailed_survey sigma slope2 poly,detailed_survey tri_num_right,recona band depth,recona reflectance,recona slope1 poly,recona slope2 poly,recona sigma band depth,recona sigma reflectance,recona sigma slope1 poly,recona sigma slope2 poly,recona tri_num_right,reconc band depth,reconc reflectance,reconc slope1 poly,reconc slope2 poly,reconc sigma band depth,reconc sigma reflectance,reconc sigma slope1 poly,reconc sigma slope2 poly,reconc tri_num,uint8_reflectance,32bit_reflectance,positions_x,positions_y,positions_z,detection_lod_level,detection_lod_code,boulder_id
u32,u32,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,u8,f32,f32,f32,f32,u8,str,u32
1,0,"""negx""",0.000676,1.002986,1.010986,1.001857,NaN,0.001074,0.00058,0.000591,NaN,25467.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,408042.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,408042.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64,0.011107,0.138693,0.13874,-0.138707,null,null,null
1,1,"""negx""",0.000681,1.002986,1.010986,1.001857,NaN,0.001074,0.00058,0.000591,NaN,25467.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,408042.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,408042.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59,0.010167,0.138667,0.138749,-0.138711,null,null,null
1,2,"""negx""",0.000654,1.002986,1.010986,1.001857,NaN,0.001074,0.00058,0.000591,NaN,25467.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,408042.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,408042.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67,0.011566,0.138643,0.138756,-0.138716,null,null,null
1,3,"""negx""",0.000647,1.002986,1.010986,1.001857,NaN,0.001074,0.00058,0.000591,NaN,25467.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,408042.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,408042.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74,0.012794,0.138617,0.138762,-0.138724,null,null,null
1,4,"""negx""",0.000632,1.002986,1.010986,1.001857,NaN,0.001074,0.00058,0.000591,NaN,25467.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,408042.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,408042.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70,0.012029,0.138591,0.138768,-0.138733,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,8187,"""posz""",0.000642,1.002942,1.013387,0.999009,NaN,0.002091,0.001041,0.001174,NaN,8957.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.0,NaN,-9999.0,142836.0,-9999.0,-9999.0,NaN,-9999.0,-9999.0,-9999.